In [19]:
# Cell 1: setup

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

project_keys = ["SERVER", "MDL", "JRACLOUD"]

mother_dir = Path("../reports/mother_files")
summary_dir = Path("../reports/summaries")
summary_dir.mkdir(parents=True, exist_ok=True)

ct_cols = {
    "CT1_Resolution_Time_Minutes": "CT1",
    "CT2_In_Progress_Minutes": "CT2",
    "CT3_Resolution_minus_Created_Minutes": "CT3",
    "CT4_Resolution_minus_FirstInProgress_Minutes": "CT4",
    "CT5_FinalDone_minus_FirstInProgress_Minutes": "CT5",
    "CT6_FirstDone_minus_FirstInProgress_Minutes": "CT6",
}

In [20]:
# Cell 2: load mother files

mother_dfs = {}

for project_key in project_keys:
    path = mother_dir / f"mother_{project_key}_2018_2020.csv"
    df = pd.read_csv(path)

    for col in ct_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    mother_dfs[project_key] = df

{project: df.shape for project, df in mother_dfs.items()}

{'SERVER': (18806, 51), 'MDL': (8521, 51), 'JRACLOUD': (5433, 51)}

In [21]:
# Cell 3: helper functions

def usable_ct_mask(s):
    s = pd.to_numeric(s, errors="coerce")
    return s.notna() & (s > 0)


def summarize_series(s, n_total):
    s = pd.to_numeric(s, errors="coerce")
    usable_mask = usable_ct_mask(s)
    usable = s.loc[usable_mask]

    out = {
        "N_total": int(n_total),
        "N_missing": int(s.isna().sum()),
        "N_nonpositive": int((s.notna() & (s <= 0)).sum()),
        "N_usable": int(usable.shape[0]),
        "Coverage_pct": round((usable.shape[0] / n_total) * 100, 2) if n_total > 0 else np.nan,
        "Median_minutes": usable.median() if not usable.empty else np.nan,
        "Q1_minutes": usable.quantile(0.25) if not usable.empty else np.nan,
        "Q3_minutes": usable.quantile(0.75) if not usable.empty else np.nan,
        "IQR_minutes": (usable.quantile(0.75) - usable.quantile(0.25)) if not usable.empty else np.nan,
        "P10_minutes": usable.quantile(0.10) if not usable.empty else np.nan,
        "P90_minutes": usable.quantile(0.90) if not usable.empty else np.nan,
        "Min_minutes": usable.min() if not usable.empty else np.nan,
        "Max_minutes": usable.max() if not usable.empty else np.nan,
        "Mean_minutes": usable.mean() if not usable.empty else np.nan,
        "Std_minutes": usable.std() if usable.shape[0] > 1 else np.nan,
    }
    return out


def add_day_columns(df, minute_cols):
    out = df.copy()
    for col in minute_cols:
        out[col.replace("_minutes", "_days")] = out[col] / (60 * 24)
    return out

In [22]:
# Cell 4: add usability flags to mother dataframes

usable_flag_cols = {ct_col: f"{ct_name}_Usable" for ct_col, ct_name in ct_cols.items()}

for project_key, df in mother_dfs.items():
    for ct_col, flag_col in usable_flag_cols.items():
        df[flag_col] = usable_ct_mask(df[ct_col])

    mother_dfs[project_key] = df

{
    project: df[[col for col in usable_flag_cols.values()]].sum().to_dict()
    for project, df in mother_dfs.items()
}

{'SERVER': {'CT1_Usable': 15963,
  'CT2_Usable': 8645,
  'CT3_Usable': 15963,
  'CT4_Usable': 9146,
  'CT5_Usable': 9169,
  'CT6_Usable': 9101},
 'MDL': {'CT1_Usable': 4896,
  'CT2_Usable': 0,
  'CT3_Usable': 4896,
  'CT4_Usable': 3463,
  'CT5_Usable': 3463,
  'CT6_Usable': 3447},
 'JRACLOUD': {'CT1_Usable': 1644,
  'CT2_Usable': 218,
  'CT3_Usable': 1644,
  'CT4_Usable': 243,
  'CT5_Usable': 265,
  'CT6_Usable': 138}}

In [23]:
# Cell 5: overwrite mother files so they carry usability flags

for project_key, df in mother_dfs.items():
    out = mother_dir / f"mother_{project_key}_2018_2020.csv"
    df.to_csv(out, index=False)

print("Mother files updated with CT usability flags.")

Mother files updated with CT usability flags.


In [24]:
# Cell 6: coverage / usability table

coverage_rows = []

for project_key, df in mother_dfs.items():
    n_total = len(df)

    for ct_col, ct_name in ct_cols.items():
        s = pd.to_numeric(df[ct_col], errors="coerce")
        usable_mask = usable_ct_mask(s)

        coverage_rows.append({
            "Project": project_key,
            "CT_Definition": ct_name,
            "CT_Column": ct_col,
            "N_total": int(n_total),
            "N_missing": int(s.isna().sum()),
            "N_nonpositive": int((s.notna() & (s <= 0)).sum()),
            "N_usable": int(usable_mask.sum()),
            "Coverage_pct": round((usable_mask.sum() / n_total) * 100, 2) if n_total > 0 else np.nan,
        })

coverage_table = (
    pd.DataFrame(coverage_rows)
    .sort_values(["Project", "CT_Definition"])
    .reset_index(drop=True)
)

coverage_table.to_csv(summary_dir / "coverage_table_usable_only.csv", index=False)
coverage_table

,Project,CT_Definition,CT_Column,N_total,N_missing,N_nonpositive,N_usable,Coverage_pct
0,JRACLOUD,CT1,CT1_Resolution_Time_Minutes,5433,0,3789,1644,30.26
1,JRACLOUD,CT2,CT2_In_Progress_Minutes,5433,0,5215,218,4.01
2,JRACLOUD,CT3,CT3_Resolution_minus_Created_Minutes,5433,3787,2,1644,30.26
3,JRACLOUD,CT4,CT4_Resolution_minus_FirstInProgress_Minutes,5433,5158,32,243,4.47
4,JRACLOUD,CT5,CT5_FinalDone_minus_FirstInProgress_Minutes,5433,5146,22,265,4.88
5,JRACLOUD,CT6,CT6_FirstDone_minus_FirstInProgress_Minutes,5433,5146,149,138,2.54
6,MDL,CT1,CT1_Resolution_Time_Minutes,8521,0,3625,4896,57.46
7,MDL,CT2,CT2_In_Progress_Minutes,8521,0,8521,0,0.00
8,MDL,CT3,CT3_Resolution_minus_Created_Minutes,8521,3620,5,4896,57.46
9,MDL,CT4,CT4_Resolution_minus_FirstInProgress_Minutes,8521,5032,26,3463,40.64


In [25]:
# Cell 7: descriptive statistics table
# All stats are calculated only on usable values (> 0)

summary_rows = []

for project_key, df in mother_dfs.items():
    n_total = len(df)

    for ct_col, ct_name in ct_cols.items():
        stats = summarize_series(df[ct_col], n_total=n_total)
        stats.update({
            "Project": project_key,
            "CT_Definition": ct_name,
            "CT_Column": ct_col,
        })
        summary_rows.append(stats)

ct_summary_table = pd.DataFrame(summary_rows)[[
    "Project", "CT_Definition", "CT_Column",
    "N_total", "N_missing", "N_nonpositive", "N_usable", "Coverage_pct",
    "Median_minutes", "Q1_minutes", "Q3_minutes", "IQR_minutes",
    "P10_minutes", "P90_minutes", "Min_minutes", "Max_minutes",
    "Mean_minutes", "Std_minutes"
]].sort_values(["Project", "CT_Definition"]).reset_index(drop=True)

ct_summary_table = add_day_columns(
    ct_summary_table,
    [
        "Median_minutes", "Q1_minutes", "Q3_minutes", "IQR_minutes",
        "P10_minutes", "P90_minutes", "Min_minutes", "Max_minutes", "Mean_minutes"
    ]
)

ct_summary_table.to_csv(summary_dir / "ct_summary_table_usable_only.csv", index=False)
ct_summary_table

,Project,CT_Definition,CT_Column,N_total,N_missing,N_nonpositive,N_usable,Coverage_pct,Median_minutes,Q1_minutes,Q3_minutes,IQR_minutes,P10_minutes,P90_minutes,Min_minutes,Max_minutes,Mean_minutes,Std_minutes,Median_days,Q1_days,Q3_days,IQR_days,P10_days,P90_days,Min_days,Max_days,Mean_days
0,JRACLOUD,CT1,CT1_Resolution_Time_Minutes,5433,0,3789,1644,30.26,86002.5,14830.75,510208.25,495377.50,1735.4,826464.0,1.0,1442755.0,270471.166058,332251.478093,59.723958,10.299132,354.311285,344.012153,1.205139,573.933333,0.000694,1001.913194,187.827199
1,JRACLOUD,CT2,CT2_In_Progress_Minutes,5433,0,5215,218,4.01,10650.0,1549.00,39917.50,38368.50,53.3,128860.3,1.0,698660.0,45083.674312,90355.365518,7.395833,1.075694,27.720486,26.644792,0.037014,89.486319,0.000694,485.180556,31.308107
2,JRACLOUD,CT3,CT3_Resolution_minus_Created_Minutes,5433,3787,2,1644,30.26,86002.5,14830.75,510209.00,495378.25,1735.4,826464.0,1.0,1442756.0,270471.177007,332251.481234,59.723958,10.299132,354.311806,344.012674,1.205139,573.933333,0.000694,1001.913889,187.827206
3,JRACLOUD,CT4,CT4_Resolution_minus_FirstInProgress_Minutes,5433,5158,32,243,4.47,12905.0,2655.00,51435.00,48780.00,101.8,181643.6,1.0,718681.0,57499.181070,112926.588079,8.961806,1.843750,35.718750,33.875000,0.070694,126.141389,0.000694,499.084028,39.929987
4,JRACLOUD,CT5,CT5_FinalDone_minus_FirstInProgress_Minutes,5433,5146,22,265,4.88,108156.0,10038.00,366590.00,356552.00,1063.6,627591.6,1.0,837755.0,216587.233962,252003.694768,75.108333,6.970833,254.576389,247.605556,0.738611,435.827500,0.000694,581.774306,150.407801
5,JRACLOUD,CT6,CT6_FirstDone_minus_FirstInProgress_Minutes,5433,5146,149,138,2.54,12804.0,1715.50,49678.50,47963.00,51.5,144902.9,1.0,465507.0,50365.934783,91001.679095,8.891667,1.191319,34.498958,33.307639,0.035764,100.627014,0.000694,323.268750,34.976344
6,MDL,CT1,CT1_Resolution_Time_Minutes,8521,0,3625,4896,57.46,27874.5,7697.00,93004.50,85307.50,1144.0,256601.0,1.0,1410121.0,94762.923407,178982.034866,19.357292,5.345139,64.586458,59.241319,0.794444,178.195139,0.000694,979.250694,65.807586
7,MDL,CT2,CT2_In_Progress_Minutes,8521,0,8521,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MDL,CT3,CT3_Resolution_minus_Created_Minutes,8521,3620,5,4896,57.46,27901.0,7697.00,93004.50,85307.50,1144.0,256631.0,1.0,1410181.0,94766.391544,178985.349538,19.375694,5.345139,64.586458,59.241319,0.794444,178.215972,0.000694,979.292361,65.809994
9,MDL,CT4,CT4_Resolution_minus_FirstInProgress_Minutes,8521,5032,26,3463,40.64,15934.0,7037.50,39481.50,32444.00,2870.6,85880.4,1.0,1299207.0,40694.055154,84437.350416,11.065278,4.887153,27.417708,22.530556,1.993472,59.639167,0.000694,902.227083,28.259761


In [26]:
# Cell 8: compact comparison table

project_compare_table = ct_summary_table[[
    "Project", "CT_Definition",
    "N_missing", "N_nonpositive", "N_usable", "Coverage_pct",
    "Median_minutes", "IQR_minutes",
    "P10_minutes", "P90_minutes",
    "Median_days", "IQR_days", "P10_days", "P90_days"
]].copy()

project_compare_table.to_csv(summary_dir / "project_compare_table_usable_only.csv", index=False)
project_compare_table

,Project,CT_Definition,N_missing,N_nonpositive,N_usable,Coverage_pct,Median_minutes,IQR_minutes,P10_minutes,P90_minutes,Median_days,IQR_days,P10_days,P90_days
0,JRACLOUD,CT1,0,3789,1644,30.26,86002.5,495377.50,1735.4,826464.0,59.723958,344.012153,1.205139,573.933333
1,JRACLOUD,CT2,0,5215,218,4.01,10650.0,38368.50,53.3,128860.3,7.395833,26.644792,0.037014,89.486319
2,JRACLOUD,CT3,3787,2,1644,30.26,86002.5,495378.25,1735.4,826464.0,59.723958,344.012674,1.205139,573.933333
3,JRACLOUD,CT4,5158,32,243,4.47,12905.0,48780.00,101.8,181643.6,8.961806,33.875000,0.070694,126.141389
4,JRACLOUD,CT5,5146,22,265,4.88,108156.0,356552.00,1063.6,627591.6,75.108333,247.605556,0.738611,435.827500
5,JRACLOUD,CT6,5146,149,138,2.54,12804.0,47963.00,51.5,144902.9,8.891667,33.307639,0.035764,100.627014
6,MDL,CT1,0,3625,4896,57.46,27874.5,85307.50,1144.0,256601.0,19.357292,59.241319,0.794444,178.195139
7,MDL,CT2,0,8521,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,MDL,CT3,3620,5,4896,57.46,27901.0,85307.50,1144.0,256631.0,19.375694,59.241319,0.794444,178.215972
9,MDL,CT4,5032,26,3463,40.64,15934.0,32444.00,2870.6,85880.4,11.065278,22.530556,1.993472,59.639167


In [27]:
# Cell 9: pivoted comparison views

usable_pivot = coverage_table.pivot(index="Project", columns="CT_Definition", values="N_usable")
coverage_pivot = coverage_table.pivot(index="Project", columns="CT_Definition", values="Coverage_pct")
median_pivot_days = ct_summary_table.pivot(index="Project", columns="CT_Definition", values="Median_days")
iqr_pivot_days = ct_summary_table.pivot(index="Project", columns="CT_Definition", values="IQR_days")

usable_pivot.to_csv(summary_dir / "usable_pivot.csv")
coverage_pivot.to_csv(summary_dir / "coverage_pivot_usable_only.csv")
median_pivot_days.to_csv(summary_dir / "median_pivot_days_usable_only.csv")
iqr_pivot_days.to_csv(summary_dir / "iqr_pivot_days_usable_only.csv")

print("Usable count")
display(usable_pivot)

print("Coverage (%)")
display(coverage_pivot)

print("Median (days)")
display(median_pivot_days)

print("IQR (days)")
display(iqr_pivot_days)

Usable count


CT_Definition,CT1,CT2,CT3,CT4,CT5,CT6
Project,,,,,,
JRACLOUD,1644,218,1644,243,265,138
MDL,4896,0,4896,3463,3463,3447
SERVER,15963,8645,15963,9146,9169,9101


Coverage (%)


CT_Definition,CT1,CT2,CT3,CT4,CT5,CT6
Project,,,,,,
JRACLOUD,30.26,4.01,30.26,4.47,4.88,2.54
MDL,57.46,0.00,57.46,40.64,40.64,40.45
SERVER,84.88,45.97,84.88,48.63,48.76,48.39


Median (days)


CT_Definition,CT1,CT2,CT3,CT4,CT5,CT6
Project,,,,,,
JRACLOUD,59.723958,7.395833,59.723958,8.961806,75.108333,8.891667
MDL,19.357292,NaN,19.375694,11.065278,11.065278,11.034028
SERVER,14.267361,1.059028,14.267361,5.694444,5.796528,5.226389


IQR (days)


CT_Definition,CT1,CT2,CT3,CT4,CT5,CT6
Project,,,,,,
JRACLOUD,344.012153,26.644792,344.012674,33.875000,247.605556,33.307639
MDL,59.241319,NaN,59.241319,22.530556,22.530556,22.502083
SERVER,43.673958,5.900694,43.673958,12.932986,13.125000,12.791667


In [28]:
# Cell 10: per-project quick view

for project_key in project_keys:
    print("=" * 80)
    print(project_key)
    display(
        ct_summary_table.loc[ct_summary_table["Project"] == project_key, [
            "CT_Definition",
            "N_usable", "Coverage_pct",
            "Median_days", "Q1_days", "Q3_days", "IQR_days",
            "P10_days", "P90_days", "Min_days", "Max_days"
        ]].sort_values("CT_Definition")
    )

SERVER


,CT_Definition,N_usable,Coverage_pct,Median_days,Q1_days,Q3_days,IQR_days,P10_days,P90_days,Min_days,Max_days
12,CT1,15963,84.88,14.267361,3.175347,46.849306,43.673958,0.470278,113.218750,0.000694,994.012500
13,CT2,8645,45.97,1.059028,0.077778,5.978472,5.900694,0.009722,14.128194,0.000694,355.336806
14,CT3,15963,84.88,14.267361,3.175347,46.849306,43.673958,0.470278,113.218750,0.000694,994.012500
15,CT4,9146,48.63,5.694444,1.240451,14.173438,12.932986,0.256597,32.860417,0.000694,777.150694
16,CT5,9169,48.76,5.796528,1.276389,14.401389,13.125000,0.262500,33.830139,0.000694,777.150694
17,CT6,9101,48.39,5.226389,1.202083,13.993750,12.791667,0.238889,31.607639,0.000694,777.150694


MDL


,CT_Definition,N_usable,Coverage_pct,Median_days,Q1_days,Q3_days,IQR_days,P10_days,P90_days,Min_days,Max_days
6,CT1,4896,57.46,19.357292,5.345139,64.586458,59.241319,0.794444,178.195139,0.000694,979.250694
7,CT2,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,CT3,4896,57.46,19.375694,5.345139,64.586458,59.241319,0.794444,178.215972,0.000694,979.292361
9,CT4,3463,40.64,11.065278,4.887153,27.417708,22.530556,1.993472,59.639167,0.000694,902.227083
10,CT5,3463,40.64,11.065278,4.887153,27.417708,22.530556,1.993472,59.639167,0.000694,902.227083
11,CT6,3447,40.45,11.034028,4.823611,27.325694,22.502083,1.909861,59.208611,0.000694,902.227083


JRACLOUD


,CT_Definition,N_usable,Coverage_pct,Median_days,Q1_days,Q3_days,IQR_days,P10_days,P90_days,Min_days,Max_days
0,CT1,1644,30.26,59.723958,10.299132,354.311285,344.012153,1.205139,573.933333,0.000694,1001.913194
1,CT2,218,4.01,7.395833,1.075694,27.720486,26.644792,0.037014,89.486319,0.000694,485.180556
2,CT3,1644,30.26,59.723958,10.299132,354.311806,344.012674,1.205139,573.933333,0.000694,1001.913889
3,CT4,243,4.47,8.961806,1.843750,35.718750,33.875000,0.070694,126.141389,0.000694,499.084028
4,CT5,265,4.88,75.108333,6.970833,254.576389,247.605556,0.738611,435.827500,0.000694,581.774306
5,CT6,138,2.54,8.891667,1.191319,34.498958,33.307639,0.035764,100.627014,0.000694,323.268750


In [29]:
# Cell 11: long-format usable issue-level export for later plots
# This is useful for the visualization notebook.

usable_long_rows = []

for project_key, df in mother_dfs.items():
    base_cols = [
        "Project_Key", "Issue_ID", "Issue_Key", "Issue_Type", "Priority",
        "Sprint_ID", "Sprint_Name", "Story_Point",
        "Created", "Resolution",
        "First_In_Progress_Timestamp", "First_Done_Timestamp", "Final_Done_Timestamp",
        "Was_Reopened"
    ]

    keep_base_cols = [col for col in base_cols if col in df.columns]

    for ct_col, ct_name in ct_cols.items():
        temp = df[keep_base_cols + [ct_col]].copy()
        temp["CT_Definition"] = ct_name
        temp["CT_Column"] = ct_col
        temp["Cycle_Time_Minutes"] = pd.to_numeric(temp[ct_col], errors="coerce")
        temp["Usable"] = usable_ct_mask(temp["Cycle_Time_Minutes"])
        temp = temp.drop(columns=[ct_col])
        usable_long_rows.append(temp)

usable_long_df = pd.concat(usable_long_rows, ignore_index=True)
usable_long_df.to_csv(summary_dir / "cycle_time_long_issue_level.csv", index=False)

usable_long_df.head()

,Project_Key,Issue_ID,Issue_Key,Issue_Type,Priority,Sprint_ID,Sprint_Name,Story_Point,Created,Resolution,First_In_Progress_Timestamp,First_Done_Timestamp,Final_Done_Timestamp,Was_Reopened,CT_Definition,CT_Column,Cycle_Time_Minutes,Usable
0,SERVER,479182,SERVER-32497,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 01:16:56,2018-01-02 17:37:10,NaN,2018-01-02 17:37:10,2018-01-29 17:29:44,0,CT1,CT1_Resolution_Time_Minutes,2420.0,True
1,SERVER,479181,SERVER-32498,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 02:20:46,2018-02-05 16:25:16,2018-01-17 19:47:07,2018-02-05 16:25:16,2018-02-05 16:25:16,0,CT1,CT1_Resolution_Time_Minutes,51244.0,True
2,SERVER,479180,SERVER-32499,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 08:57:08,2018-01-03 04:25:52,NaN,2018-01-03 04:25:52,2018-01-03 04:25:52,0,CT1,CT1_Resolution_Time_Minutes,2608.0,True
3,SERVER,479179,SERVER-32500,Task,Major - P3,NaN,NaN,NaN,2018-01-01 17:23:11,2018-01-01 18:32:41,NaN,2018-01-01 18:32:41,2018-01-29 17:30:19,0,CT1,CT1_Resolution_Time_Minutes,69.0,True
4,SERVER,479178,SERVER-32501,Improvement,Major - P3,NaN,NaN,NaN,2018-01-02 05:27:53,2018-01-15 05:23:57,2018-01-05 00:12:28,2018-01-15 05:23:57,2018-01-15 05:23:57,0,CT1,CT1_Resolution_Time_Minutes,18716.0,True


In [30]:
# Cell 12: optional usable-only long export for plotting

usable_long_only_df = usable_long_df.loc[usable_long_df["Usable"]].copy()
usable_long_only_df.to_csv(summary_dir / "cycle_time_long_issue_level_usable_only.csv", index=False)

usable_long_only_df.head()

,Project_Key,Issue_ID,Issue_Key,Issue_Type,Priority,Sprint_ID,Sprint_Name,Story_Point,Created,Resolution,First_In_Progress_Timestamp,First_Done_Timestamp,Final_Done_Timestamp,Was_Reopened,CT_Definition,CT_Column,Cycle_Time_Minutes,Usable
0,SERVER,479182,SERVER-32497,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 01:16:56,2018-01-02 17:37:10,NaN,2018-01-02 17:37:10,2018-01-29 17:29:44,0,CT1,CT1_Resolution_Time_Minutes,2420.0,True
1,SERVER,479181,SERVER-32498,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 02:20:46,2018-02-05 16:25:16,2018-01-17 19:47:07,2018-02-05 16:25:16,2018-02-05 16:25:16,0,CT1,CT1_Resolution_Time_Minutes,51244.0,True
2,SERVER,479180,SERVER-32499,Bug,Major - P3,NaN,NaN,NaN,2018-01-01 08:57:08,2018-01-03 04:25:52,NaN,2018-01-03 04:25:52,2018-01-03 04:25:52,0,CT1,CT1_Resolution_Time_Minutes,2608.0,True
3,SERVER,479179,SERVER-32500,Task,Major - P3,NaN,NaN,NaN,2018-01-01 17:23:11,2018-01-01 18:32:41,NaN,2018-01-01 18:32:41,2018-01-29 17:30:19,0,CT1,CT1_Resolution_Time_Minutes,69.0,True
4,SERVER,479178,SERVER-32501,Improvement,Major - P3,NaN,NaN,NaN,2018-01-02 05:27:53,2018-01-15 05:23:57,2018-01-05 00:12:28,2018-01-15 05:23:57,2018-01-15 05:23:57,0,CT1,CT1_Resolution_Time_Minutes,18716.0,True
